# 🧩 Python, NumPy & Pandas — The Data Backbone of ML

Every machine learning model — from a simple linear regressor to a transformer — ultimately consumes **numbers arranged in arrays**. Before we touch any algorithm, we need to be fluent in the two libraries that hold ML's entire data layer together:

- **NumPy** → fast, vectorized numerical arrays (the "tensor" before deep learning made that word popular)
- **Pandas** → labeled, tabular data (the "spreadsheet" that talks to Python)

This notebook builds intuition + math for both, then ties them together in a mini end-to-end example.

📖 Full mathematical explanation: [README.md](README.md)


## 1. Why NumPy? — Vectorization vs Loops

A Python `list` stores **pointers to objects** scattered in memory. A NumPy `ndarray` stores **raw numbers in one contiguous memory block**, so operations run as compiled C loops (SIMD-vectorized) instead of Python's slow bytecode loop.

Mathematically, for two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^n$, element-wise addition is:

$$\mathbf{c}_i = \mathbf{a}_i + \mathbf{b}_i \quad \forall i \in \{1, ..., n\}$$

NumPy computes all $n$ additions in one vectorized instruction — no explicit Python `for` loop needed.


In [1]:
import numpy as np
import pandas as pd
import time

# --- Speed comparison: pure Python loop vs NumPy vectorized op ---
n = 1_000_000
py_list_a = list(range(n))
py_list_b = list(range(n))
np_arr_a = np.arange(n)
np_arr_b = np.arange(n)

# Pure Python: element-wise add via list comprehension (interpreted, one object at a time)
start = time.time()
py_result = [a + b for a, b in zip(py_list_a, py_list_b)]
py_time = time.time() - start
print(f"Pure Python loop : {py_time:.4f} sec")


Pure Python loop : 0.0526 sec


In [2]:
# NumPy: same operation, vectorized in compiled C
start = time.time()
np_result = np_arr_a + np_arr_b
np_time = time.time() - start

print(f"NumPy vectorized : {np_time:.4f} sec")
print(f"Speedup          : {py_time / np_time:.1f}x faster")


NumPy vectorized : 0.0026 sec
Speedup          : 20.0x faster


## 2. Array Creation & Attributes

Every `ndarray` has three attributes that matter constantly in ML code:

| Attribute | Meaning | ML relevance |
|---|---|---|
| `shape` | dimensions, e.g. `(batch, features)` | must match between layers/operations |
| `dtype` | data type, e.g. `float32` | controls memory + precision (deep learning defaults to `float32`) |
| `ndim` | number of axes | scalar=0, vector=1, matrix=2, tensor=3+ |


In [3]:
# Common creation patterns you'll reuse constantly in ML pipelines
zeros = np.zeros((3, 4))          # placeholder tensor, e.g. initializing an output buffer
ones = np.ones((2, 2))            # useful for bias initialization or masks
identity = np.eye(3)              # identity matrix I — used in regularization (Ridge: X^T X + λI)
ranged = np.arange(0, 10, 2)      # like Python range(), but returns an array
linear = np.linspace(0, 1, 5)     # 5 evenly spaced points in [0, 1] — great for probability grids
random_uniform = np.random.rand(2, 3)     # uniform [0, 1) — common weight init baseline
print("identity matrix:\n", identity)
print("linspace:", linear)


identity matrix:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
linspace: [0.   0.25 0.5  0.75 1.  ]


In [4]:
sample = np.array([[1, 2, 3], [4, 5, 6]])
print("shape:", sample.shape)   # (2 rows, 3 cols) -> (batch_size=2, features=3) in ML terms
print("dtype:", sample.dtype)
print("ndim :", sample.ndim)
print("size :", sample.size)    # total elements = shape product


shape: (2, 3)
dtype: int64
ndim : 2
size : 6


## 3. Indexing, Slicing & Boolean Masking

NumPy indexing generalizes Python's `list[start:stop:step]` to N dimensions, and adds **boolean masking** — the mechanism behind almost every "filter my dataset" operation in ML preprocessing.

Given condition $f(x)$, boolean masking selects the subset:

$$S = \{ x_i \mid f(x_i) = \text{True} \}$$


In [5]:
matrix = np.arange(1, 13).reshape(3, 4)   # reshape flat 12 elements into 3x4
print("Full matrix:\n", matrix)

print("\nRow 1:            ", matrix[1])          # second row
print("Column 2:         ", matrix[:, 2])          # third column, all rows
print("Sub-block (top-left 2x2):\n", matrix[:2, :2])


Full matrix:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Row 1:             [5 6 7 8]
Column 2:          [ 3  7 11]
Sub-block (top-left 2x2):
 [[1 2]
 [5 6]]


In [6]:
# Boolean masking — the core operation behind "filter rows where condition is true"
mask = matrix > 6
print("Mask (matrix > 6):\n", mask)
print("Values where matrix > 6:", matrix[mask])     # flattened array of matching values

# Fancy indexing: select specific rows by index list
print("\nRows [0, 2]:\n", matrix[[0, 2]])


Mask (matrix > 6):
 [[False False False False]
 [False False  True  True]
 [ True  True  True  True]]
Values where matrix > 6: [ 7  8  9 10 11 12]

Rows [0, 2]:
 [[ 1  2  3  4]
 [ 9 10 11 12]]


## 4. Broadcasting — Operating on Mismatched Shapes

Broadcasting lets NumPy apply operations between arrays of **different shapes** without manually copying data, following two rules:

1. Compare shapes from the **rightmost** dimension.
2. Two dimensions are compatible if they are **equal**, or one of them is **1**.

This is exactly how you normalize a feature matrix $X \in \mathbb{R}^{n \times d}$ by a per-feature mean vector $\mu \in \mathbb{R}^{d}$:

$$X_{\text{norm}}[i, j] = X[i, j] - \mu[j]$$

without writing an explicit loop over rows $i$.


In [7]:
X = np.array([[10, 200, 3000],
              [20, 400, 6000],
              [30, 600, 9000]], dtype=float)   # 3 samples, 3 features

mean = X.mean(axis=0)   # axis=0 -> collapse rows, one mean PER COLUMN (per feature)
std = X.std(axis=0)

print("Per-feature mean:", mean)
print("Per-feature std :", std)


Per-feature mean: [  20.  400. 6000.]
Per-feature std : [   8.16496581  163.29931619 2449.48974278]


In [8]:
# Broadcasting: (3,3) array minus (3,) vector -> vector is "stretched" across every row
X_standardized = (X - mean) / std
print("Standardized X (zero mean, unit variance per feature):\n", X_standardized)


Standardized X (zero mean, unit variance per feature):
 [[-1.22474487 -1.22474487 -1.22474487]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  1.22474487]]


## 5. Aggregation with `axis` — The Most Confused Concept in NumPy

`axis=0` collapses **rows** (result has one value per column). `axis=1` collapses **columns** (result has one value per row). Think of `axis` as **"which dimension disappears."**

For a matrix $X \in \mathbb{R}^{n \times d}$:

$$\text{mean}(X, \text{axis}=0)_j = \frac{1}{n}\sum_{i=1}^{n} X_{ij} \quad \text{(per-feature mean, used in normalization)}$$
$$\text{mean}(X, \text{axis}=1)_i = \frac{1}{d}\sum_{j=1}^{d} X_{ij} \quad \text{(per-sample mean, e.g. average pixel intensity)}$$


In [9]:
print("Column-wise sum (axis=0):", X.sum(axis=0))   # one sum per feature/column
print("Row-wise sum    (axis=1):", X.sum(axis=1))   # one sum per sample/row


Column-wise sum (axis=0): [   60.  1200. 18000.]
Row-wise sum    (axis=1): [3210. 6420. 9630.]


In [10]:
# Linear algebra: dot product and matrix multiplication — the core op of every neural network layer
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("Dot product a·b:", np.dot(a, b))           # scalar: sum(a_i * b_i)

W = np.random.randn(3, 2)   # weight matrix, e.g. mapping 3 input features -> 2 outputs
print("\nMatrix multiply X @ W (forward pass style):\n", X @ W)


Dot product a·b: 32

Matrix multiply X @ W (forward pass style):
 [[ 5493.57198421  2052.04984967]
 [10987.14396842  4104.09969935]
 [16480.71595263  6156.14954902]]


## 6. Random Numbers — Reproducibility Matters in ML

`np.random.seed()` fixes the pseudo-random generator's starting state, so every run of your notebook produces **identical** train/test splits, weight initializations, and shuffle orders — critical for debugging and fair experiment comparison.


In [11]:
np.random.seed(42)   # fixing the seed = reproducible experiments

# Simulating a train/test split index shuffle (the mechanism behind train_test_split)
indices = np.arange(10)
np.random.shuffle(indices)
print("Shuffled indices:", indices)


Shuffled indices: [8 1 5 0 7 2 9 4 3 6]


In [12]:
split_point = int(0.8 * len(indices))
train_idx, test_idx = indices[:split_point], indices[split_point:]
print("Train indices:", train_idx)
print("Test indices :", test_idx)


Train indices: [8 1 5 0 7 2 9 4]
Test indices : [3 6]


## 7. Pandas — Labeled Data on Top of NumPy

A Pandas **`Series`** is a NumPy array + an index (labels). A **`DataFrame`** is a dict of aligned `Series` — conceptually a table where every column is its own labeled array.

$$\text{DataFrame} = \{ \text{col}_1: \text{Series}_1,\ \text{col}_2: \text{Series}_2,\ \ldots \}$$


In [13]:
# Series: 1D labeled array
grades = pd.Series([88, 92, 79, 95], index=["Alice", "Bob", "Carol", "Dave"])
print(grades)
print("\nAlice's grade:", grades["Alice"])   # label-based lookup, unlike plain NumPy arrays


Alice    88
Bob      92
Carol    79
Dave     95
dtype: int64

Alice's grade: 88


In [14]:
# DataFrame: dict of columns
df = pd.DataFrame({
    "student": ["Alice", "Bob", "Carol", "Dave"],
    "math": [88, 92, 79, 95],
    "science": [91, 85, 88, 99],
    "attendance_pct": [95, 100, 80, 90],
})
df


,student,math,science,attendance_pct
0,Alice,88,91,95
1,Bob,92,85,100
2,Carol,79,88,80
3,Dave,95,99,90


## 8. `loc` vs `iloc` — Label-based vs Position-based Access

- **`loc`** → select by **label/condition** (inclusive of both endpoints in slices)
- **`iloc`** → select by **integer position** (like plain NumPy, exclusive end)

Mixing these up is the #1 source of silent bugs in Pandas code.


In [15]:
print("iloc[0] (first row by position):\n", df.iloc[0])
print("\nloc[0, 'math'] (row label 0, column 'math'):", df.loc[0, "math"])


iloc[0] (first row by position):
 student           Alice
math                 88
science              91
attendance_pct       95
Name: 0, dtype: object

loc[0, 'math'] (row label 0, column 'math'): 88


In [16]:
# Conditional filtering — the Pandas equivalent of NumPy boolean masking
high_performers = df.loc[df["math"] > 85]
print("Students with math > 85:\n", high_performers)


Students with math > 85:
   student  math  science  attendance_pct
0   Alice    88       91              95
1     Bob    92       85             100
3    Dave    95       99              90


## 9. Missing Data — `isna`, `fillna`, `dropna`

Real datasets are never clean. Pandas represents missing values as `NaN` (Not a Number), and gives three core tools to handle them before any algorithm can consume the data (most ML algorithms cannot handle `NaN` natively).


In [17]:
df_dirty = df.copy()
df_dirty.loc[1, "science"] = np.nan   # simulate a missing value
df_dirty.loc[3, "attendance_pct"] = np.nan

print("Missing value mask:\n", df_dirty.isna())
print("\nRows with any missing value dropped:\n", df_dirty.dropna())


Missing value mask:
    student   math  science  attendance_pct
0    False  False    False           False
1    False  False     True           False
2    False  False    False           False
3    False  False    False            True

Rows with any missing value dropped:
   student  math  science  attendance_pct
0   Alice    88     91.0            95.0
2   Carol    79     88.0            80.0


In [18]:
# Mean imputation — replace NaN with the column's mean (a common, simple strategy)
df_filled = df_dirty.fillna(df_dirty.mean(numeric_only=True))
print("After mean imputation:\n", df_filled)


After mean imputation:
   student  math    science  attendance_pct
0   Alice    88  91.000000       95.000000
1     Bob    92  92.666667      100.000000
2   Carol    79  88.000000       80.000000
3    Dave    95  99.000000       91.666667


## 10. GroupBy — Split-Apply-Combine

`groupby` implements the **split-apply-combine** pattern: split rows into groups by a key, apply an aggregation to each group independently, then combine results into one table. Formally, for groups $G_1, ..., G_k$ partitioning the data:

$$\text{result}_k = f(\{x_i \mid x_i \in G_k\})$$


In [19]:
df["pass_math"] = df["math"] >= 85   # derived boolean column

grouped = df.groupby("pass_math")["science"].mean()
print("Average science score, grouped by math pass/fail:\n", grouped)


Average science score, grouped by math pass/fail:
 pass_math
False    88.000000
True     91.666667
Name: science, dtype: float64


In [20]:
# describe() gives count, mean, std, min, quartiles, max in one call — the fastest EDA sanity check
df.describe()


,math,science,attendance_pct
count,4.000000,4.000000,4.000000
mean,88.500000,90.750000,91.250000
std,6.952218,6.020797,8.539126
min,79.000000,85.000000,80.000000
25%,85.750000,87.250000,87.500000
50%,90.000000,89.500000,92.500000
75%,92.750000,93.000000,96.250000
max,95.000000,99.000000,100.000000


## 11. Mini Practical Example — NumPy + Pandas Together

We simulate a tiny "dataset" the way you'd receive one in practice: a NumPy feature matrix (from a sensor/model) that we wrap in a Pandas DataFrame for labeled exploration.


In [21]:
np.random.seed(0)

# Simulate 100 samples, 3 features, generated as if from an experiment/sensor (NumPy's job)
raw_features = np.random.randn(100, 3) * [10, 5, 2] + [50, 20, 5]
feature_names = ["temperature", "humidity", "pressure_delta"]

# Wrap in a DataFrame for labeled inspection and EDA (Pandas' job)
sensor_df = pd.DataFrame(raw_features, columns=feature_names)
sensor_df["reading_valid"] = sensor_df["pressure_delta"] > 4  # derived label

print(sensor_df.describe())


       temperature    humidity  pressure_delta
count   100.000000  100.000000      100.000000
mean     51.005205   20.531262        4.751632
std      10.312031    4.934869        1.961802
min      27.765968    6.137036       -0.105980
25%      45.158997   16.805781        3.373598
50%      50.576037   20.664296        4.717485
75%      57.113213   23.870712        6.126390
max      73.831448   31.519583        9.518618


In [22]:
print("Valid vs invalid reading counts:\n", sensor_df["reading_valid"].value_counts())
print("\nMean temperature by validity:\n", sensor_df.groupby("reading_valid")["temperature"].mean())


Valid vs invalid reading counts:
 reading_valid
True     61
False    39
Name: count, dtype: int64

Mean temperature by validity:
 reading_valid
False    49.148385
True     52.192353
Name: temperature, dtype: float64


## 12. Sorting & Unique Values

`np.sort` returns a sorted copy; `np.argsort` returns the **indices** that would sort the array — often more useful, since you can use those indices to reorder a second, related array (e.g. sort labels by score). `np.unique` removes duplicates and can also return counts.


In [23]:
scores = np.array([88, 72, 95, 60, 88, 72])
names = np.array(["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"])

sorted_scores = np.sort(scores)
print("Sorted scores:", sorted_scores)

order = np.argsort(scores)[::-1]   # descending rank order
print("Rank order (highest first):", names[order])


Sorted scores: [60 72 72 88 88 95]
Rank order (highest first): ['Carol' 'Eve' 'Alice' 'Frank' 'Bob' 'Dave']


In [24]:
unique_scores, counts = np.unique(scores, return_counts=True)
print("Unique scores:", unique_scores)
print("Counts:        ", counts)


Unique scores: [60 72 88 95]
Counts:         [1 2 2 1]


## 13. Conditional Logic — `np.where` and `np.select`

`np.where(condition, if_true, if_false)` is a vectorized ternary operator — the NumPy equivalent of writing an `if/else` inside a loop, but applied to an entire array at once. `np.select` extends this to multiple conditions/branches (like a vectorized `elif` chain).


In [25]:
temps = np.array([15, 22, 30, 5, 18, 35])

# Two-way branch: label each temperature as hot or cold
labels = np.where(temps >= 25, "hot", "cold")
print("Two-way labels:", labels)


Two-way labels: ['cold' 'cold' 'hot' 'cold' 'cold' 'hot']


In [26]:
# Multi-way branch: cold / mild / hot using np.select
conditions = [temps < 15, (temps >= 15) & (temps < 25), temps >= 25]
choices = ["cold", "mild", "hot"]
labels_multi = np.select(conditions, choices, default="unknown")   # default must share a dtype with string choices
print("Three-way labels:", labels_multi)


Three-way labels: ['mild' 'mild' 'hot' 'cold' 'mild' 'hot']


## 14. Set Operations

NumPy provides set algebra directly on arrays — useful for tasks like finding which customer IDs appear in both this month's and last month's data.


In [27]:
group_a = np.array([1, 2, 3, 4, 5])
group_b = np.array([4, 5, 6, 7, 8])

print("Intersection (in both):", np.intersect1d(group_a, group_b))
print("Union (in either):     ", np.union1d(group_a, group_b))


Intersection (in both): [4 5]
Union (in either):      [1 2 3 4 5 6 7 8]


In [28]:
print("In A but not B:        ", np.setdiff1d(group_a, group_b))
print("Is 3 in group_b?       ", np.isin(3, group_b))


In A but not B:         [1 2 3]
Is 3 in group_b?        False


## 15. Merging DataFrames — `pd.merge`

Real-world data rarely lives in one table. `pd.merge` implements SQL-style joins — **inner** (only matching keys), **left** (all left rows, matched right rows or `NaN`), **outer** (all rows from both, `NaN` where unmatched) — the mechanism behind combining, e.g., a customers table with an orders table.


In [29]:
students = pd.DataFrame({"student_id": [1, 2, 3, 4], "name": ["Alice", "Bob", "Carol", "Dave"]})
grades = pd.DataFrame({"student_id": [1, 2, 2, 5], "subject": ["Math", "Math", "Science", "Math"], "grade": [88, 92, 79, 60]})

inner = students.merge(grades, on="student_id", how="inner")
print("Inner join (only matched):\n", inner)


Inner join (only matched):
    student_id   name  subject  grade
0           1  Alice     Math     88
1           2    Bob     Math     92
2           2    Bob  Science     79


In [30]:
left = students.merge(grades, on="student_id", how="left")
outer = students.merge(grades, on="student_id", how="outer")

print("Left join (all students, unmatched -> NaN):\n", left)
print("\nOuter join (everything, unmatched -> NaN both sides):\n", outer)


Left join (all students, unmatched -> NaN):
    student_id   name  subject  grade
0           1  Alice     Math   88.0
1           2    Bob     Math   92.0
2           2    Bob  Science   79.0
3           3  Carol      NaN    NaN
4           4   Dave      NaN    NaN

Outer join (everything, unmatched -> NaN both sides):
    student_id   name  subject  grade
0           1  Alice     Math   88.0
1           2    Bob     Math   92.0
2           2    Bob  Science   79.0
3           3  Carol      NaN    NaN
4           4   Dave      NaN    NaN
5           5    NaN     Math   60.0


## 16. Pivot Tables — Reshaping for Summary

`pd.pivot_table` turns long-format data (one row per observation) into a wide summary table (rows × columns × aggregated values) — the Pandas equivalent of an Excel pivot table, and a fast way to compare a metric across two categorical dimensions at once.


In [31]:
sales = pd.DataFrame({
    "region": ["North", "North", "South", "South", "North", "South"],
    "product": ["A", "B", "A", "B", "A", "A"],
    "revenue": [100, 150, 200, 130, 120, 180],
})
sales


,region,product,revenue
0,North,A,100
1,North,B,150
2,South,A,200
3,South,B,130
4,North,A,120
5,South,A,180


In [32]:
pivot = pd.pivot_table(sales, values="revenue", index="region", columns="product", aggfunc="sum")
print(pivot)


product    A    B
region           
North    220  150
South    380  130


## 17. String Methods & `apply()` — Custom Transformations

The `.str` accessor vectorizes Python string operations across a whole column (no manual loop needed). `.apply()` runs any custom function element-wise — the escape hatch for transformations no built-in method covers.


In [33]:
names_df = pd.DataFrame({"full_name": [" Alice Smith", "bob jones ", "CAROL WHITE"]})

names_df["clean_name"] = names_df["full_name"].str.strip().str.title()   # vectorized string cleaning
names_df["first_name"] = names_df["clean_name"].str.split().str[0]
names_df


,full_name,clean_name,first_name
0,Alice Smith,Alice Smith,Alice
1,bob jones,Bob Jones,Bob
2,CAROL WHITE,Carol White,Carol


In [34]:
names_df["initials"] = names_df["clean_name"].apply(lambda n: "".join(word[0] for word in n.split()))
names_df


,full_name,clean_name,first_name,initials
0,Alice Smith,Alice Smith,Alice,AS
1,bob jones,Bob Jones,Bob,BJ
2,CAROL WHITE,Carol White,Carol,CW


## ✅ Key Takeaways

- NumPy arrays are fast because they're **contiguous, typed, and vectorized** — avoid Python `for` loops on numeric data whenever a vectorized alternative exists.
- **Broadcasting** and **`axis`** are the two ideas that unlock 90% of NumPy's expressive power — internalize them early.
- Pandas `Series`/`DataFrame` = NumPy arrays + labels; `loc` is label-based, `iloc` is position-based.
- Missing-value handling (`isna`/`fillna`/`dropna`) and `groupby` are non-negotiable EDA tools you'll use on every real dataset.
- `pd.merge` (SQL-style joins) and `pd.pivot_table` are how you combine and reshape multi-table real-world data — rarely does one dataset arrive as a single clean table.
- Matrix multiplication (`@` / `np.dot`) is the literal operation every neural network layer performs — this notebook's linear algebra section is not incidental, it's the foundation.

**Next up:** [02_Data_Visualization](../02_Data_Visualization/) — turning these arrays and tables into plots.
